# 00. Coding Agents

Coding Agent 是当前 Agent 技术最成熟的落地场景。以 SWE-agent 为代表，它通过 **Agent-Computer Interface (ACI)** 设计让 LLM 像人类开发者一样操作：

- 读文件（`view`）
- 搜索代码（`grep`）
- 编辑文件（`edit`）
- 运行测试（`test`）
- 提交修复（`submit`）

> **检查点**：能说出 SWE-agent 的 ACI 设计为什么比「把所有代码塞进 prompt」更好。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# mini coding agent 核心逻辑
import os, sys, tempfile, subprocess

def setup_sample_repo() -> str:
    """创建一个带 bug 的示例仓库"""
    tmp = tempfile.mkdtemp(prefix="coding_agent_")
    with open(os.path.join(tmp, "calc.py"), "w") as f:
        f.write("def add(a,b): return a-b  # BUG: should be +\ndef mul(a,b): return a*b\n")
    with open(os.path.join(tmp, "test_calc.py"), "w") as f:
        f.write("from calc import add, mul\nassert add(2,3)==5, f'got {add(2,3)}'\nassert mul(4,5)==20\nprint('OK')\n")
    return tmp

def read_file(path: str) -> str:
    with open(path) as f: return f.read()

def find_issues(content: str) -> list[dict]:
    """规则引擎：发现代码中的 bug 模式"""
    issues = []
    for i, line in enumerate(content.split('\n'), 1):
        if 'return a-b' in line and 'BUG' in line:
            issues.append({"line":i,"type":"wrong_op","fix":"return a+b","desc":"add() uses - instead of +"})
    return issues

def apply_fix(repo: str, filename: str, old: str, new: str) -> bool:
    path = os.path.join(repo, filename)
    content = read_file(path)
    if old in content:
        with open(path, "w") as f:
            f.write(content.replace(old, new))
        return True
    return False

def run_tests(repo: str) -> tuple[bool, str]:
    r = subprocess.run([sys.executable, os.path.join(repo,"test_calc.py")],
                       capture_output=True, text=True, cwd=repo)
    return r.returncode == 0, r.stdout.strip()

# 执行
dir = setup_sample_repo()
print(f"Repo: {dir}\n")
print("1. Reading calc.py...")
code = read_file(os.path.join(dir, "calc.py"))
print(f"   Found:\n{code}")

issues = find_issues(code)
print(f"2. Issues found: {len(issues)}")
for iss in issues:
    print(f"   Line {iss['line']}: {iss['desc']} → {iss['fix']}")

for iss in issues:
    ok = apply_fix(dir, "calc.py", "return a-b  # BUG: should be +", "return a+b  # FIXED")
    print(f"3. Fixed: {'✓' if ok else '✗'}")

passed, output = run_tests(dir)
print(f"4. Tests: {'✓ PASS' if passed else '✗ FAIL'} — {output}")

import shutil; shutil.rmtree(dir)

Repo: /tmp/coding_agent_wnvt08q7

1. Reading calc.py...
   Found:
def add(a,b): return a-b  # BUG: should be +
def mul(a,b): return a*b

2. Issues found: 1
   Line 1: add() uses - instead of + → return a+b
3. Fixed: ✓
4. Tests: ✓ PASS — OK


## SWE-bench 评估体系

SWE-bench 的核心创新：用**真实 GitHub issue + 对应测试**来衡量 Agent 的代码修复能力。

- SWE-bench Verified: 500 个经过人工验证的高质量 issue
- 评估指标：resolved rate（测试通过率）
- mini-SWE-agent: ~100 行 Python，65% resolved rate

> **检查点**：能解释为什么 SWE-bench 的「真实 issue + 测试」比「生成代码片段」更有意义。

## 真实 API 实验

真实 LLM 驱动的 mini coding agent（`mini_coding_agent.py --use-api`）。


In [3]:
SCRIPT = ROOT / 'agents' / 'frontiers' / 'scripts' / 'mini_coding_agent.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--task', 'find-bug', '--use-api', '--verbose'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


Repo: /tmp/mini_swe_up0ch4q_

Code:
def add(a, b):
    return a - b  # BUG: should be a + b

def multiply(a, b):
    return a * b

if __name__ == "__main__":
    print(f"2+3 = {add(2, 3)}")
    print(f"4*5 = {multiply(4, 5)}")


Issues found [LLM]: 1
  The add function incorrectly subtracts instead of adding.
  ✓ Applied:     return a - b  # BUG: should be a + b... →     return a + b  # FIX: corrected to ad...

✓ PASS: OK



## 练习

1. 给 mini coding agent 增加 `search` 工具（grep 功能）：按关键词搜索代码，返回匹配行。
2. 设计一个更难的任务：不仅修复 bug，还要添加一个新功能（并写测试）。
3. 思考 coding agent 的安全边界：如果 Agent 可以执行任意 bash 命令，最小权限应该怎么设计？